# Directed results analysis

Examines the outputs of `query_directions_without_proto.py`
(`results/directional_resolved_without_proto/`). Run that script first - this notebook only reads its CSVs.

Sections: load -> overview -> sanity checks -> expert-edge comparison -> cross-context agreement ->
LLM tie-broken pairs.

In [ ]:
from pathlib import Path
from itertools import combinations

import pandas as pd

ROOT = Path("..").resolve()

RESULT_DIR = ROOT / "results/directional_resolved_without_proto"
CONTEXTS = ["kg_llm", "llm", "rag"]
METRICS = ["plausibility", "association", "temporality"]
# column holding the boolean prediction in each resolved CSV
METRIC_COL = {
    "plausibility": "Plausibility",
    "association": "Association",
    "temporality": "Temporality",
}

In [ ]:
# load whatever resolved CSVs exist; report anything missing instead of crashing
results = {}  # (context, metric) -> DataFrame
missing = []
for context in CONTEXTS:
    for metric in METRICS:
        path = RESULT_DIR / f"{context}_{metric}_without_proto_resolved.csv"
        if path.exists():
            results[(context, metric)] = pd.read_csv(path)
        else:
            missing.append(str(path.relative_to(ROOT)))

print(f"loaded {len(results)} resolved result files")
if missing:
    print(f"missing {len(missing)} (run query_directions_without_proto.py to produce them):")
    for m in missing:
        print("  -", m)

## Overview

Edge counts per configuration, and how many of those directions came from the LLM tie-break
(`Direction_Resolved == True`) versus falling out of the undirected predictions directly.

In [ ]:
overview = pd.DataFrame([
    {
        "context": context,
        "metric": metric,
        "edges": len(df),
        "tie_broken": int((df["Direction_Resolved"] == True).sum()),
        "tie_broken_pct": round(100 * (df["Direction_Resolved"] == True).mean(), 1),
    }
    for (context, metric), df in results.items()
])
overview.sort_values(["metric", "context"]).reset_index(drop=True)

## Sanity checks

After resolution each output should contain **at most one direction per unordered pair** and no
duplicate rows. Anything listed here indicates a resolution bug.

In [ ]:
problems = []
for key, df in results.items():
    edges = set(zip(df["Var1"], df["Var2"]))
    contradictions = sorted({tuple(sorted(e)) for e in edges if (e[1], e[0]) in edges})
    dupes = int(df.duplicated(subset=["Var1", "Var2"]).sum())
    if contradictions or dupes:
        problems.append({"config": key, "contradictions": contradictions, "duplicate_rows": dupes})

if problems:
    display(pd.DataFrame(problems))
else:
    print("OK: no bidirectional contradictions and no duplicate (Var1, Var2) rows in any output")

## Comparison against expert edges

Confusion-matrix metrics of each configuration's directed edge set against
`data/expert_edge_pairs.csv`: **TP / FP / FN / TN, precision, TPR (recall), FDR, FPR, F1**.
An edge only counts as a true positive if the *direction* matches the expert edge.

The negative class needs a universe: we use every directed pair the pipeline actually queried
(`data/full_cleaned.csv`, both orientations, with the Sleep rename applied) - TN is the candidate
pairs that neither the model nor the experts assert.

In [ ]:
expert = pd.read_csv(ROOT / "data/expert_edge_pairs.csv").replace("Sleep", "Sleep disturbance")
expert_edges = set(zip(expert["Var1"], expert["Var2"]))

# candidate universe: every directed pair the pipeline queried, in both orientations
# (resolution can emit the direction opposite to the queried row)
full = pd.read_csv(ROOT / "data/full_cleaned.csv").drop(columns=["Unnamed: 0"]).replace("Sleep", "Sleep disturbance")
universe = set(zip(full["var1"], full["var2"]))
universe |= {(b, a) for a, b in universe}

print(f"{len(expert_edges)} expert directed edges, {len(universe)} candidate directed pairs")
outside = expert_edges - universe
if outside:
    print(f"note: {len(outside)} expert edge(s) outside the queried candidate set (count as FN, never TN): {sorted(outside)}")

def confusion_scores(predicted):
    predicted = predicted & universe
    tp = len(predicted & expert_edges)
    fp = len(predicted - expert_edges)
    fn = len(expert_edges - predicted)
    tn = len(universe - predicted - expert_edges)
    precision = tp / (tp + fp) if tp + fp else 0.0
    tpr = tp / (tp + fn) if tp + fn else 0.0   # recall / sensitivity
    fdr = fp / (tp + fp) if tp + fp else 0.0   # 1 - precision
    fpr = fp / (fp + tn) if fp + tn else 0.0
    f1 = 2 * precision * tpr / (precision + tpr) if precision + tpr else 0.0
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": round(precision, 3), "TPR": round(tpr, 3),
        "FDR": round(fdr, 3), "FPR": round(fpr, 3), "F1": round(f1, 3),
    }

scores = pd.DataFrame([
    {"context": context, "metric": metric, "edges": len(df),
     **confusion_scores(set(zip(df["Var1"], df["Var2"])))}
    for (context, metric), df in results.items()
])
scores.sort_values(["metric", "F1"], ascending=[True, False]).reset_index(drop=True)

## Agreement between contexts

For each setting/metric, Jaccard overlap of the directed edge sets produced by the three contexts
(kg_llm vs llm vs rag). Low agreement means the knowledge source is driving the directions, not the data.

In [ ]:
agreement_rows = []
for metric in METRICS:
    for ctx_a, ctx_b in combinations(CONTEXTS, 2):
        key_a, key_b = (ctx_a, metric), (ctx_b, metric)
        if key_a not in results or key_b not in results:
            continue
        edges_a = set(zip(results[key_a]["Var1"], results[key_a]["Var2"]))
        edges_b = set(zip(results[key_b]["Var1"], results[key_b]["Var2"]))
        union = edges_a | edges_b
        agreement_rows.append({
            "metric": metric,
            "pair": f"{ctx_a} vs {ctx_b}",
            "common": len(edges_a & edges_b),
            "jaccard": round(len(edges_a & edges_b) / len(union), 3) if union else 1.0,
        })

pd.DataFrame(agreement_rows)

## LLM tie-broken pairs

The pairs that were predicted in both directions and needed the direction prompt to break the tie.
Frequency across configurations shows which relationships are persistently ambiguous; the reasoning
column of one configuration is shown for spot-checking (rows whose reasoning is a bare error message
are failed resolutions that fell back to alphabetical order - see the run's WARNING output).

In [ ]:
tie_broken = pd.concat(
    [
        df[df["Direction_Resolved"] == True].assign(context=context, metric=metric)
        for (context, metric), df in results.items()
        if (df["Direction_Resolved"] == True).any()
    ],
    ignore_index=True,
) if results else pd.DataFrame()

if len(tie_broken):
    freq = (
        tie_broken.assign(pair=tie_broken.apply(lambda r: tuple(sorted([r["Var1"], r["Var2"]])), axis=1))
        .groupby("pair").size().sort_values(ascending=False).rename("times_tie_broken")
    )
    display(freq.to_frame())
else:
    print("no tie-broken pairs in the loaded results")

In [ ]:
# spot-check the resolved directions and reasoning for one configuration
INSPECT = ("kg_llm", "plausibility")

if INSPECT in results:
    df = results[INSPECT]
    resolved = df[df["Direction_Resolved"] == True]
    metric_reasoning_col = f"{METRIC_COL[INSPECT[1]]} Reasoning"
    with pd.option_context("display.max_colwidth", 200):
        display(resolved[["Var1", "Var2", metric_reasoning_col]])
else:
    print(f"{INSPECT} not loaded")